In [6]:
import numpy as np
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
import re
import time


# STEP 1: LOAD MODEL
print("Loading pre-trained Word2Vec/GloVe model...")
model = api.load('glove-twitter-25')
print("Model loaded successfully!")


# STEP 2: PREPROCESS FUNCTION
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()


# STEP 3: SENTENCE EMBEDDING
def get_sentence_vector(words, model):
    word_vectors = [model[word] for word in words if word in model]
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(word_vectors, axis=0)


# STEP 4: DATASET INDEXING
faq_dataset = [
    "How can I change my account password?",
    "Which laptops are best for programming and coding?",
    "The system freezes when I open the application.",
    "I need to upgrade my current subscription plan.",
    "How do I check the delivery status of my order?"
]

print("\n" + "="*80)
print("STEP 1: DATASET INDEXING (Generating Sentence Embeddings)")
print("="*80)

dataset_vectors = []

for i, sentence in enumerate(faq_dataset):
    processed = preprocess(sentence)
    vector = get_sentence_vector(processed, model)
    dataset_vectors.append(vector)
    print(f"Index {i}: {sentence[:50]}... -> Vector Size: {vector.shape}")


# STEP 5: SEMANTIC SEARCH
queries = [
    "unable to login account",
    "best computers for coding",
    "application not responding",
    "how to see my order status",
    "upgrade membership plan"
]

print("\n" + "="*80)
print("STEP 2: REAL-TIME SEMANTIC SEARCH EVALUATION")
print("="*80)

print(f"{'User Query':<30} | {'Top Semantic Match':<40} | {'Score'}")
print("-"*80)

for q_text in queries:
    q_processed = preprocess(q_text)
    q_vector = get_sentence_vector(q_processed, model).reshape(1, -1)

    sims = []
    for v in dataset_vectors:
        sim = cosine_similarity(q_vector, v.reshape(1, -1))[0][0]
        sims.append(sim)

    best_idx = np.argmax(sims)


    match_text = faq_dataset[best_idx]
    if len(match_text) > 38:
        match_text = match_text[:35] + "..."

    print(f"{q_text:<30} | {match_text:<40} | {sims[best_idx]:.4f}")


# STEP 6: WORD SIMILARITY
print("\n" + "="*80)
print("STEP 3: WORD VECTOR SIMILARITY ANALYSIS (PROVING THEORY)")
print("="*80)

word_pairs = [
    ('laptop', 'computer'),
    ('password', 'login'),
    ('crash', 'freeze')
]

for w1, w2 in word_pairs:
    if w1 in model and w2 in model:
        sim = model.similarity(w1, w2)
        print(f"Similarity between '{w1}' and '{w2}': {sim:.4f}")


# STEP 7: ANALYSIS SUMMARY
print("\n" + "="*80)
print("ANALYSIS SUMMARY")
print("="*80)

print("The results demonstrate that word embeddings capture contextual meaning.")
print("Synonyms (like 'computer' and 'laptop') show high similarity scores,")
print("allowing the search system to work without exact keyword matching.")
print("="*80)

Loading pre-trained Word2Vec/GloVe model...
Model loaded successfully!

STEP 1: DATASET INDEXING (Generating Sentence Embeddings)
Index 0: How can I change my account password?... -> Vector Size: (25,)
Index 1: Which laptops are best for programming and coding?... -> Vector Size: (25,)
Index 2: The system freezes when I open the application.... -> Vector Size: (25,)
Index 3: I need to upgrade my current subscription plan.... -> Vector Size: (25,)
Index 4: How do I check the delivery status of my order?... -> Vector Size: (25,)

STEP 2: REAL-TIME SEMANTIC SEARCH EVALUATION
User Query                     | Top Semantic Match                       | Score
--------------------------------------------------------------------------------
unable to login account        | I need to upgrade my current subscr...   | 0.9151
best computers for coding      | Which laptops are best for programm...   | 0.9852
application not responding     | The system freezes when I open the ...   | 0.8983
how to se